In [1]:
pip install sentence-transformers scikit-learn numpy supabase python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install groq -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import json
import time
import asyncio

import numpy as np
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from supabase import create_client
from groq import Groq

In [4]:
import os
from dotenv import load_dotenv
from supabase import ClientOptions

# Baca kredensial dari file .env (bukan hardcode -> tidak gampang dicabut & aman saat di-push).
load_dotenv()
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_KEY"]   # pipeline butuh akses penuh (service_role)

# Timeout client dinaikkan (default rawan 'read operation timed out' saat query besar/lambat).
_opsi_supabase = ClientOptions(postgrest_client_timeout=120, storage_client_timeout=120)
supabase = create_client(SUPABASE_URL, SUPABASE_KEY, options=_opsi_supabase)

In [5]:
import os

# Beberapa Groq API key dari .env (dipisah koma). Tidak ada lagi key hardcoded di notebook.
GROQ_API_KEYS = [k.strip() for k in os.environ.get("GROQ_API_KEYS", "").split(",") if k.strip()]
if not GROQ_API_KEYS:
    raise RuntimeError("GROQ_API_KEYS kosong — pastikan file .env ada & terisi di folder ini.")

_panjang_aneh = [i for i, k in enumerate(GROQ_API_KEYS) if len(k) != 56]
if _panjang_aneh:
    print(f"⚠️  Key indeks {_panjang_aneh} panjangnya bukan 56 char -> kemungkinan tidak sah (401)!")

MODEL_LLM = "llama-3.1-8b-instant"
MAKS_KATA_REQUEST = 2500  # Batas kata konteks agar total token tetap di bawah limit per request Groq

current_key_idx = 0
client = Groq(api_key=GROQ_API_KEYS[current_key_idx])


def switch_groq_key():
    """Memutar (rolling) ke API Key Groq berikutnya saat key aktif terkena limit."""
    global current_key_idx, client
    current_key_idx = (current_key_idx + 1) % len(GROQ_API_KEYS)
    client = Groq(api_key=GROQ_API_KEYS[current_key_idx])
    print(f"🔄 [API ROLLING] Beralih ke API Key Index: {current_key_idx}")

In [6]:
print("[INIT] Memuat model embedding...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

[INIT] Memuat model embedding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Preprocessing + Clustering (Worker 1)

In [7]:
# ── Knob clustering (semua parameter tuning di satu tempat) ──
# Nilai DATA-DRIVEN dari Eval_Clustering (tuner replika Worker 3, grid bobot x threshold,
# diuji di 3 jendela waktu 1/3/7 hari). Objektif = silhouette x coverage; parameter di bawah
# adalah yang skornya paling STABIL lintas-window (skor_rata2 ~0.252, sil ~0.41, coverage ~61%).
# Riwayat: BOBOT 0.6->0.45 (isi lebih berperan), ATAS 0.72->0.75 & BAWAH 0.60->0.63 (sweet spot terukur).
BOBOT_JUDUL = 0.45       # bobot judul pada vektor gabungan (sisanya utk isi)
THRESHOLD_ATAS = 0.75    # cosine >= ini -> langsung gabung
THRESHOLD_BAWAH = 0.63   # zona abu-abu [bawah, atas): gabung hanya bila overlap entitas memadai
NER_MIN = 0.30           # ambang minimal overlap entitas di zona abu-abu

STOP_WORDS_ENTITAS = frozenset({
    'di', 'ke', 'dari', 'pada', 'dalam', 'yaitu', 'untuk', 'yang', 'dan', 'ini', 'itu',
    'telah', 'sebuah', 'puluhan', 'ratusan', 'insiden', 'skuad', 'pasukan', 'tim',
    'berkat', 'setelah', 'menyusul', 'pecahan', 'sejumlah', 'kepala', 'beberapa',
    'banyak', 'surat', 'para', 'warga', 'calon', 'getaran', 'bencana', 'masyarakat',
    'korban', 'aksi', 'gelar', 'pawai', 'ri', 'the', 'akan', 'istana', 'presiden',
})


def bersihkan_teks_struktural(teks):
    """Menormalkan spasi/whitespace berlebih menjadi satu spasi."""
    return re.sub(r'\s+', ' ', str(teks)).strip()


def ekstrak_entitas_cepat(teks):
    """Mengekstrak entitas (kata berkapital) dari teks, mengabaikan stop words."""
    entitas = re.findall(r'\b[A-Z][a-zA-Z0-9]*\b', str(teks))
    return {e.lower() for e in entitas if e.lower() not in STOP_WORDS_ENTITAS}


def overlap_similarity(set_artikel, set_klaster):
    """Rasio irisan entitas artikel terhadap total entitas artikel (0.0 - 1.0)."""
    if not set_artikel:
        return 0.0
    return len(set_artikel & set_klaster) / len(set_artikel)


def _normalisasi_l2(vektor):
    """Bagi vektor dengan norma L2 (aman dari pembagian nol)."""
    norma = np.linalg.norm(vektor)
    return vektor / norma if norma > 0 else vektor


def vektorisasi_berbobot(judul, isi):
    """Vektor gabungan judul (BOBOT_JUDUL) + isi (1-BOBOT_JUDUL), dinormalisasi L2.
    SEMUA representasi (artikel maupun seed centroid klaster) WAJIB lewat fungsi ini
    agar berada di ruang vektor yang sama -> cosine matching akurat."""
    teks_judul = bersihkan_teks_struktural(judul)
    teks_isi = bersihkan_teks_struktural(" ".join(str(isi).split()[:150]))

    if teks_isi:
        # Satu panggilan encode untuk kedua teks (lebih cepat daripada dua panggilan terpisah).
        vec_judul, vec_isi = embed_model.encode([teks_judul, teks_isi])
    else:
        vec_judul = embed_model.encode([teks_judul])[0]
        vec_isi = vec_judul

    vektor_kombinasi = (vec_judul * BOBOT_JUDUL) + (vec_isi * (1.0 - BOBOT_JUDUL))
    return _normalisasi_l2(vektor_kombinasi)


def vektorisasi_berbobot_batch(pasangan):
    """Versi BATCH dari vektorisasi_berbobot: encode SEKALI untuk banyak (judul, isi).
    Hasil vektor IDENTIK dgn memanggil vektorisasi_berbobot satu-satu (math sama: bobot
    judul/isi + L2), hanya jauh lebih cepat -> dipakai Worker 1 untuk seed & artikel."""
    judul_list, isi_list = [], []
    for judul, isi in pasangan:
        tj = bersihkan_teks_struktural(judul)
        ti = bersihkan_teks_struktural(" ".join(str(isi).split()[:150]))
        judul_list.append(tj)
        isi_list.append(ti if ti else tj)   # isi kosong -> vec_isi = vec_judul (sama spt versi single)
    Vj = np.asarray(embed_model.encode(judul_list, batch_size=64, show_progress_bar=False))
    Vi = np.asarray(embed_model.encode(isi_list, batch_size=64, show_progress_bar=False))
    V = Vj * BOBOT_JUDUL + Vi * (1.0 - BOBOT_JUDUL)
    norma = np.linalg.norm(V, axis=1, keepdims=True)
    norma[norma == 0] = 1.0
    return V / norma

In [8]:
def worker_1_clustering():
    print("\n[WORKER 1] Memulai Preprocessing + Smart Clustering (Running-Centroid Mode)...")

    res_berita = supabase.table("tabel_berita") \
        .select("id_berita, judul, isi_teks") \
        .eq("status_proses", 0) \
        .execute()
    berita_baru = res_berita.data

    if not berita_baru:
        print("[WORKER 1] Tidak ada antrean berita baru.")
        return

    # Hanya pertimbangkan klaster aktif dalam 24 jam terakhir.
    waktu_batas = (datetime.now(ZoneInfo("Asia/Jakarta")) - timedelta(hours=24)).isoformat()
    res_klaster = supabase.table("tabel_cluster") \
        .select("id_cluster, summary_text, judul_summary, jumlah_berita") \
        .gte("waktu_terbentuk", waktu_batas) \
        .execute()

    # ── OPTIMASI N+1: artikel pertama untuk klaster TANPA ringkasan ditarik BATCH ──
    # (Dulu satu query per-klaster -> ratusan round-trip -> 'read operation timed out'.)
    # Chunk 100 id agar URL .in_ tetap aman; paginasi range bila >1000 baris per chunk.
    id_tanpa_ringkasan = [c["id_cluster"] for c in res_klaster.data
                          if not ((c.get("judul_summary") or "") or (c.get("summary_text") or ""))]
    artikel_pertama = {}
    for i in range(0, len(id_tanpa_ringkasan), 100):
        chunk = id_tanpa_ringkasan[i:i + 100]
        mulai = 0
        while True:
            res_b = supabase.table("tabel_berita") \
                .select("id_berita, judul, isi_teks, id_cluster") \
                .in_("id_cluster", chunk) \
                .order("id_berita", desc=False) \
                .range(mulai, mulai + 999) \
                .execute()
            for a in res_b.data:
                idc = a["id_cluster"]
                if idc not in artikel_pertama:   # urut id asc -> kemunculan pertama = artikel pertama
                    artikel_pertama[idc] = a
            if len(res_b.data) < 1000:
                break
            mulai += 1000

    # Susun seed dalam URUTAN ASLI res_klaster (jaga tie-breaking argmax tetap sama), lalu
    # ENCODE SEKALI via vektorisasi_berbobot_batch (hasil identik dgn vektorisasi_berbobot).
    seed_meta, seed_pasangan, seed_teks_entitas = [], [], []
    for c in res_klaster.data:
        id_c = c["id_cluster"]
        judul_ref = c.get("judul_summary") or ""
        isi_ref = c.get("summary_text") or ""
        if judul_ref or isi_ref:
            judul_seed, isi_seed = judul_ref, isi_ref
            teks_ent = f"{judul_ref} {isi_ref}"
        else:
            a = artikel_pertama.get(id_c)
            if not a:
                continue   # klaster tanpa artikel (anomali) -> lewati, sama seperti versi lama
            judul_seed, isi_seed = a["judul"], a["isi_teks"]
            teks_ent = f"{a['judul']} {a['isi_teks']}"
        seed_meta.append((id_c, max(1, int(c.get("jumlah_berita") or 1))))
        seed_pasangan.append((judul_seed, isi_seed))
        seed_teks_entitas.append(teks_ent)

    memori_klaster = {}
    if seed_pasangan:
        seed_vektor = vektorisasi_berbobot_batch(seed_pasangan)   # encode sekali utk semua seed
        for (id_c, n_seed), vec_k, teks_ent in zip(seed_meta, seed_vektor, seed_teks_entitas):
            # Bobotkan seed sebanyak jumlah anggota lama -> centroid stabil (cegah drift/over-segmentasi).
            memori_klaster[id_c] = {
                "sum": vec_k * n_seed,
                "count": n_seed,
                "vektor": vec_k,
                "entitas": ekstrak_entitas_cepat(teks_ent),
            }

    # Pra-encode SEMUA vektor artikel baru sekali (penugasan tetap sekuensial krn running-centroid).
    vektor_baru = vektorisasi_berbobot_batch([(b["judul"], b["isi_teks"]) for b in berita_baru])

    pembaruan_klaster = {}
    klaster_direvisi = set()
    klaster_baru_count = 0

    for idx, b in enumerate(berita_baru):
        id_berita = b["id_berita"]
        vektor_artikel = vektor_baru[idx]
        entitas_artikel = ekstrak_entitas_cepat(b["isi_teks"])

        id_terbaik = None

        if memori_klaster:
            daftar_id = list(memori_klaster.keys())
            matriks_klaster = np.array([memori_klaster[i]["vektor"] for i in daftar_id])

            skor_semantik = cosine_similarity([vektor_artikel], matriks_klaster)[0]
            indeks_max = int(np.argmax(skor_semantik))
            skor_max = skor_semantik[indeks_max]
            kandidat_id = daftar_id[indeks_max]

            if skor_max >= THRESHOLD_ATAS:
                id_terbaik = kandidat_id
            elif skor_max >= THRESHOLD_BAWAH:
                # Zona abu-abu: validasi dengan kemiripan entitas (NER overlap).
                skor_ner = overlap_similarity(entitas_artikel, memori_klaster[kandidat_id]["entitas"])
                if skor_ner >= NER_MIN:
                    id_terbaik = kandidat_id

        if id_terbaik is not None:
            target_id = id_terbaik
            klaster_direvisi.add(id_terbaik)
            # Perbarui centroid sebagai RATA-RATA BERJALAN (bukan dikunci ke artikel pertama).
            m = memori_klaster[id_terbaik]
            m["sum"] = m["sum"] + vektor_artikel
            m["count"] += 1
            m["vektor"] = _normalisasi_l2(m["sum"])
            m["entitas"].update(entitas_artikel)
        else:
            res_new = supabase.table("tabel_cluster").insert({"status_summary": 0, "status_sentimen": 0, "status_prediksi": 0}).execute()
            target_id = res_new.data[0]["id_cluster"]
            memori_klaster[target_id] = {
                "sum": vektor_artikel.copy(),
                "count": 1,
                "vektor": vektor_artikel,
                "entitas": entitas_artikel,
            }
            klaster_baru_count += 1

        pembaruan_klaster.setdefault(target_id, []).append(id_berita)

    # Tetapkan klaster + tandai berita sudah diproses.
    for id_c, list_id_b in pembaruan_klaster.items():
        supabase.table("tabel_berita").update({
            "id_cluster": id_c,
            "status_proses": 1
        }).in_("id_berita", list_id_b).execute()

    # Klaster yang menerima berita baru perlu dirangkum & diprediksi ulang.
    if klaster_direvisi:
        daftar_id_revisi = list(klaster_direvisi)
        supabase.table("tabel_sentimen_aktor").delete().in_("id_cluster", daftar_id_revisi).execute()
        supabase.table("tabel_sektor").delete().in_("id_cluster", daftar_id_revisi).execute()
        supabase.table("tabel_cluster").update({
            "status_summary": 0,
            "status_sentimen": 0,
            "status_prediksi": 0
        }).in_("id_cluster", daftar_id_revisi).execute()

    # ── OPTIMASI N+1: jumlah_berita dari tally running-centroid (in-memory), bukan query COUNT
    #    per-klaster. Otoritatif: seed dimulai dari jumlah_berita lama + increment tiap artikel masuk.
    for id_c in pembaruan_klaster:
        supabase.table("tabel_cluster") \
            .update({"jumlah_berita": memori_klaster[id_c]["count"]}) \
            .eq("id_cluster", id_c).execute()

    print(f"[WORKER 1] Selesai. Klaster Baru: {klaster_baru_count}, Klaster Direvisi: {len(klaster_direvisi)}")


# Worker 2 — Summarizer

Helper parsing JSON LLM (dipakai bersama Worker 2, 3, & 4).

In [9]:
def bersihkan_markdown_json(teks_kotor):
    """
    Menghapus pembungkus markdown (```json ... ```) yang kadang disertakan LLM,
    sehingga output dapat langsung di-parse json.loads tanpa JSONDecodeError.
    """
    teks = re.sub(r'^```json\s*', '', teks_kotor, flags=re.MULTILINE)
    teks = re.sub(r'^```\s*', '', teks, flags=re.MULTILINE)
    return teks.strip()


def normalisasi_persentase(nilai, default=50):
    """Mengubah persentase keluaran LLM menjadi integer aman pada rentang 0-100."""
    try:
        angka = int(round(float(str(nilai).replace("%", "").strip())))
    except (ValueError, TypeError):
        return default
    return max(0, min(100, angka))


def ambil_satu_kalimat(teks, maks_char=255):
    """Mengambil 1 kalimat pertama dari teks alasan dan memotong panjang berlebih."""
    teks = bersihkan_teks_struktural(teks)
    if not teks:
        return "-"
    kalimat_pertama = re.split(r'(?<=[.!?])\s', teks, maxsplit=1)[0]
    return kalimat_pertama[:maks_char]

In [10]:
def worker_2_summarizer():
    print("\n[WORKER 2] Memulai Summarization...")

    res_klaster = supabase.table("tabel_cluster") \
        .select("id_cluster") \
        .eq("status_summary", 0) \
        .gte("jumlah_berita", 5) \
        .execute()

    antrean_klaster = res_klaster.data
    if not antrean_klaster:
        print("[WORKER 2] Tidak ada klaster matang (>= 5 berita) dalam 24 jam terakhir untuk dirangkum.")
        return

    print(f"[WORKER 2] Ditemukan {len(antrean_klaster)} klaster untuk dirangkum.\n")

    for baris in antrean_klaster:
        id_cluster = baris["id_cluster"]

        # Ambil sampel berita dari ujung awal & akhir klaster, lalu dedup berdasarkan judul.
        res_awal = supabase.table("tabel_berita") \
            .select("judul, isi_teks") \
            .eq("id_cluster", id_cluster) \
            .order("id_berita", desc=False) \
            .limit(15) \
            .execute()

        res_akhir = supabase.table("tabel_berita") \
            .select("judul, isi_teks") \
            .eq("id_cluster", id_cluster) \
            .order("id_berita", desc=True) \
            .limit(15) \
            .execute()

        berita_unik = list({b["judul"]: b for b in (res_awal.data + res_akhir.data)}.values())
        jumlah_artikel = len(berita_unik)

        # Semakin banyak artikel, semakin pendek potongan tiap artikel agar muat dalam limit token.
        if jumlah_artikel > 15:
            batas_kata = 100
        elif jumlah_artikel > 5:
            batas_kata = 150
        else:
            batas_kata = 250

        potongan = [
            f"Judul: {b['judul']} | Isi: {' '.join(str(b['isi_teks']).split()[:batas_kata])}."
            for b in berita_unik if b["isi_teks"]
        ]
        teks_final = " ".join(" ".join(potongan).split()[:MAKS_KATA_REQUEST])

        prompt = f"""Ringkas kumpulan potongan berita berikut (satu peristiwa) dan keluarkan JSON murni.

BERITA:
"{teks_final}"

Aturan:
1. judul: 1 kalimat gaya jurnalistik, inti seluruh peristiwa, 4-10 kata.
2. rangkuman: 4-15 kalimat (WAJIB minimal 4). Pakai struktur PIRAMIDA TERBALIK: kalimat-kalimat awal sudah memuat inti 5W1H (Apa, Siapa, Di mana, Kapan, Mengapa, Bagaimana) secara padat & jelas, detail pendukung menyusul makin ke bawah. Bila ada >1 peristiwa tak nyambung, pisahkan jadi beberapa paragraf.

Format: {{"judul":"...","rangkuman":"..."}}"""

        pesan_error = ""
        berhasil = False

        for _ in range(len(GROQ_API_KEYS)):
            try:
                waktu_mulai = time.time()
                response = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt}],
                    model=MODEL_LLM,
                    temperature=0.2,
                    response_format={"type": "json_object"}
                )

                hasil_json = json.loads(bersihkan_markdown_json(response.choices[0].message.content))
                judul_ai = hasil_json.get("judul", "Tanpa Judul")
                rangkuman_ai = hasil_json.get("rangkuman", "Gagal menghasilkan rangkuman.")

                # Sentimen aktor lama dibersihkan -> Worker 3 akan mengisi ulang dari ringkasan baru ini.
                supabase.table("tabel_sentimen_aktor").delete().eq("id_cluster", id_cluster).execute()
                supabase.table("tabel_cluster").update({
                    "judul_summary": judul_ai,
                    "summary_text": rangkuman_ai,
                    "status_summary": 1
                }).eq("id_cluster", id_cluster).execute()

                waktu_eksekusi = round(time.time() - waktu_mulai, 2)
                print(f"  [V] Klaster {id_cluster} Sukses | Token: {response.usage.total_tokens} | Waktu: {waktu_eksekusi}s")
                berhasil = True
                time.sleep(3)
                break

            except Exception as e:
                pesan_error = str(e).lower()

                if "request too large" in pesan_error:
                    print(f"  [X] Klaster {id_cluster} ditolak: teks melebihi limit token per request. Menandai gagal.")
                    try:
                        supabase.table("tabel_cluster").update({"status_summary": 2}).eq("id_cluster", id_cluster).execute()
                    except Exception:
                        pass
                    break

                elif "per day" in pesan_error or "tpd" in pesan_error:
                    print(f"     [!] Limit HARIAN habis pada API Key {current_key_idx}. Memutar API Key...")
                    switch_groq_key()

                elif "per minute" in pesan_error or "tpm" in pesan_error or "rate limit reached" in pesan_error:
                    print(f"     [!] Limit PER MENIT penuh pada API Key {current_key_idx}. Menunggu 60 detik...")
                    time.sleep(60)

                else:
                    print(f"  [X] Error Sistem/Parsing pada klaster {id_cluster}: {pesan_error}")
                    try:
                        supabase.table("tabel_cluster").update({"status_summary": 2}).eq("id_cluster", id_cluster).execute()
                    except Exception:
                        pass
                    break

        if not berhasil and ("429" in pesan_error or "413" in pesan_error or "rate limit" in pesan_error):
            print("     [!] SEMUA API Key terkena limit. Menunggu 60 detik agar limit tereset...")
            time.sleep(60)

    print("\n[WORKER 2] Seluruh antrean ringkasan selesai diproses.")


# Worker 3 — Sentimen

In [11]:
def worker_3_sentimen():
    print("\n[WORKER 3] Memulai Sentiment Analysis (aktor diambil dari ringkasan Worker 2)...")

    # Klaster yang sudah punya ringkasan (status_summary=1) tapi belum diprediksi (status_prediksi=0)
    # = antrean yang butuh sentimen. Dijalankan SEBELUM Worker 4 dalam satu siklus, jadi sentimen
    # selalu siap saat prediksi membacanya. Tanpa kolom status baru (skema DB tak diubah).
    res_klaster = supabase.table("tabel_cluster") \
        .select("id_cluster, judul_summary, summary_text") \
        .eq("status_summary", 1) \
        .eq("status_sentimen", 0) \
        .execute()

    antrean_klaster = res_klaster.data
    if not antrean_klaster:
        print("[WORKER 3] Tidak ada klaster yang perlu dianalisis sentimennya.")
        return

    print(f"[WORKER 3] Ditemukan {len(antrean_klaster)} klaster untuk dianalisis sentimennya.\n")

    for baris in antrean_klaster:
        id_cluster = baris["id_cluster"]
        judul_ai = baris.get("judul_summary", "")
        rangkuman_ai = baris.get("summary_text", "")

        if not rangkuman_ai:
            print(f" -> Klaster {id_cluster} dilewati (belum ada ringkasan).")
            continue

        prompt = f"""Analisis sentimen aktor dari ringkasan berita berikut. Keluarkan JSON murni.

JUDUL: {judul_ai}
RINGKASAN: {rangkuman_ai}

Aturan:
1. aktor: maks 3 tokoh/instansi UTAMA yang disebut di ringkasan. Abaikan figuran (pengamat/saksi). Boleh <3.
2. tiap aktor: sentimen WAJIB hanya "Positif" atau "Negatif" (tidak ada Netral; pilih yang paling dominan), persentase (bulat 0-100 = kekuatan sentimen), alasan (tepat 1 kalimat).

Format: {{"aktor":[{{"nama":"...","sentimen":"Positif/Negatif","persentase":85,"alasan":"..."}}]}}"""

        pesan_error = ""
        berhasil = False

        for _ in range(len(GROQ_API_KEYS)):
            try:
                waktu_mulai = time.time()
                response = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt}],
                    model=MODEL_LLM,
                    temperature=0.2,
                    response_format={"type": "json_object"}
                )

                hasil_json = json.loads(bersihkan_markdown_json(response.choices[0].message.content))
                daftar_aktor = hasil_json.get("aktor", [])

                supabase.table("tabel_sentimen_aktor").delete().eq("id_cluster", id_cluster).execute()

                if daftar_aktor:
                    data_aktor = [
                        {
                            "id_cluster": id_cluster,
                            "nama_aktor": str(aktor.get("nama", "Anonim"))[:100],
                            # Biner: apa pun selain "positif" (termasuk jika model nekat "Netral") -> Negatif.
                            "sentimen": "Positif" if str(aktor.get("sentimen", "")).strip().lower().startswith("pos") else "Negatif",
                            "persentase": normalisasi_persentase(aktor.get("persentase")),
                            "alasan": ambil_satu_kalimat(aktor.get("alasan", "-")),
                        }
                        for aktor in daftar_aktor
                    ]
                    supabase.table("tabel_sentimen_aktor").insert(data_aktor).execute()

                supabase.table("tabel_cluster").update({"status_sentimen": 1}).eq("id_cluster", id_cluster).execute()
                waktu_eksekusi = round(time.time() - waktu_mulai, 2)
                print(f"  [V] Klaster {id_cluster} Sukses | {len(daftar_aktor)} aktor | Token: {response.usage.total_tokens} | Waktu: {waktu_eksekusi}s")
                berhasil = True
                time.sleep(3)
                break

            except Exception as e:
                pesan_error = str(e).lower()

                if "request too large" in pesan_error:
                    print(f"  [X] Klaster {id_cluster} ditolak: teks melebihi limit token per request. Dilewati.")
                    break

                elif "per day" in pesan_error or "tpd" in pesan_error:
                    print(f"     [!] Limit HARIAN habis pada API Key {current_key_idx}. Memutar API Key...")
                    switch_groq_key()

                elif "per minute" in pesan_error or "tpm" in pesan_error or "rate limit reached" in pesan_error:
                    print(f"     [!] Limit PER MENIT penuh pada API Key {current_key_idx}. Menunggu 60 detik...")
                    time.sleep(60)

                else:
                    print(f"  [X] Error Sistem/Parsing pada klaster {id_cluster}: {pesan_error}")
                    break

        if not berhasil and ("429" in pesan_error or "413" in pesan_error or "rate limit" in pesan_error):
            print("     [!] SEMUA API Key terkena limit. Menunggu 60 detik agar limit tereset...")
            time.sleep(60)

    print("\n[WORKER 3] Seluruh antrean sentimen selesai diproses.")


# Worker 4 — Prediksi Dampak

In [12]:
def worker_4_prediksi():
    print("\n[WORKER 4] Memulai Prediksi Dampak (Mode Relasional Tabel Sektor)...")

    try:
        res_klaster = supabase.table("tabel_cluster") \
            .select("id_cluster, judul_summary, summary_text") \
            .eq("status_prediksi", 0) \
            .eq("status_summary", 1) \
            .execute()
        antrean_klaster = res_klaster.data
    except Exception as e:
        print(f"[WORKER 4] Gagal menarik antrean klaster: {e}")
        return

    if not antrean_klaster:
        print("[WORKER 4] Tidak ada cluster yang perlu diprediksi.")
        return

    print(f"[WORKER 4] Menemukan {len(antrean_klaster)} cluster untuk diprediksi.\n")

    for baris in antrean_klaster:
        id_cluster = baris["id_cluster"]
        judul_ai = baris.get("judul_summary", "")
        rangkuman_ai = baris.get("summary_text", "")

        if not rangkuman_ai:
            print(f" -> Cluster {id_cluster} dilewati (tidak ada ringkasan).")
            continue

        try:
            res_aktor = supabase.table("tabel_sentimen_aktor") \
                .select("nama_aktor, sentimen, persentase") \
                .eq("id_cluster", id_cluster) \
                .execute()
            konteks_aktor = "".join(
                f"- {a['nama_aktor']}: {a['sentimen']} ({a['persentase']}%)\n"
                for a in res_aktor.data
            )
        except Exception:
            konteks_aktor = ""

        prompt_prediksi = f"""Kamu analis kebijakan publik, ekonomi, dan sosial Indonesia. Keluarkan JSON murni.

JUDUL: {judul_ai}
RINGKASAN: {rangkuman_ai}
SENTIMEN AKTOR:
{konteks_aktor if konteks_aktor else "Tidak ada data aktor."}

Tugas: berdasarkan fakta di ringkasan (jangan mengarang), tentukan sektor yang paling mungkin terdampak dalam 1-4 minggu ke depan. Untuk tiap sektor isi:
- nama_sektor: nama sektor terdampak (mis. Ekonomi, Politik, Hukum, Sosial, Pendidikan, Kesehatan, Lingkungan).
- prediksi_dampak: 1 kalimat dampak yang mungkin + siapa yang terdampak + dampak spesifik (positif/negatif).
- tingkat_risiko: Tinggi / Sedang / Rendah.

Format: {{"analisis_sektor":[{{"nama_sektor":"...","prediksi_dampak":"...","tingkat_risiko":"Tinggi/Sedang/Rendah"}}]}}"""

        pesan_error = ""
        berhasil = False

        for _ in range(len(GROQ_API_KEYS)):
            try:
                response_prediksi = client.chat.completions.create(
                    messages=[{"role": "user", "content": prompt_prediksi}],
                    model=MODEL_LLM,
                    temperature=0.3,
                    response_format={"type": "json_object"}
                )

                hasil_prediksi = json.loads(bersihkan_markdown_json(response_prediksi.choices[0].message.content))
                daftar_sektor = hasil_prediksi.get("analisis_sektor", [])

                if daftar_sektor:
                    data_insert = []
                    for s in daftar_sektor:
                        risiko = str(s.get("tingkat_risiko", "Sedang")).strip().capitalize()
                        if risiko not in ("Tinggi", "Sedang", "Rendah"):
                            risiko = "Sedang"
                        data_insert.append({
                            "id_cluster": id_cluster,
                            "nama_sektor": s.get("nama_sektor", "Lainnya"),
                            "prediksi_dampak": s.get("prediksi_dampak", "-"),
                            "tingkat_risiko": risiko
                        })

                    supabase.table("tabel_sektor").delete().eq("id_cluster", id_cluster).execute()
                    supabase.table("tabel_sektor").insert(data_insert).execute()

                supabase.table("tabel_cluster").update({"status_prediksi": 1}).eq("id_cluster", id_cluster).execute()
                print(f"  [V] Prediksi Cluster {id_cluster} tersimpan ({len(daftar_sektor)} Sektor diekstrak).")

                berhasil = True
                time.sleep(3)
                break

            except Exception as e:
                pesan_error = str(e).lower()

                if "request too large" in pesan_error:
                    print(f"  [X] Klaster {id_cluster} ditolak: teks melebihi limit token per request. Menandai gagal.")
                    try:
                        supabase.table("tabel_cluster").update({"status_prediksi": 2}).eq("id_cluster", id_cluster).execute()
                    except Exception:
                        pass
                    break

                elif "per day" in pesan_error or "tpd" in pesan_error:
                    print(f"     [!] Limit HARIAN habis pada API Key {current_key_idx}. Memutar API Key...")
                    switch_groq_key()
                    time.sleep(2)

                elif "per minute" in pesan_error or "tpm" in pesan_error or "rate limit reached" in pesan_error:
                    print(f"     [!] Limit PER MENIT penuh pada API Key {current_key_idx}. Menunggu 60 detik...")
                    time.sleep(60)

                else:
                    print(f"  [X] Error Sistem/Parsing pada klaster {id_cluster}: {pesan_error}")
                    try:
                        supabase.table("tabel_cluster").update({"status_prediksi": 2}).eq("id_cluster", id_cluster).execute()
                    except Exception:
                        pass
                    break

        if not berhasil and ("429" in pesan_error or "413" in pesan_error or "rate limit" in pesan_error):
            print("     [!] Semua API Key terkena limit di Worker 4. Menunggu 60 detik...")
            time.sleep(60)

    print("\n[WORKER 4] Seluruh antrean prediksi selesai diproses.")


# Pipeline Orkestrasi

In [13]:
async def run_ai_pipeline_periodically():
    interval_detik = 2700  # 45 menit

    while True:
        waktu_mulai = time.time()

        print(f"\n{'='*60}")
        print(f"🚀 [SIKLUS AI DIMULAI: {time.strftime('%H:%M:%S')}]")
        print(f"{'='*60}")

        try:
            print("\n[PROSES 1/4] Menjalankan Worker 1 (Preprocessing + Smart Clustering)...")
            await asyncio.to_thread(worker_1_clustering)

            print("\n[PROSES 2/4] Menjalankan Worker 2 (Groq AI Summarizer)...")
            await asyncio.to_thread(worker_2_summarizer)

            print("\n[PROSES 3/4] Menjalankan Worker 3 (Groq AI Sentiment Analysis)...")
            await asyncio.to_thread(worker_3_sentimen)

            print("\n[PROSES 4/4] Menjalankan Worker 4 (Prediksi Dampak)...")
            await asyncio.to_thread(worker_4_prediksi)

        except Exception as e:
            print(f"\n❌ [ERROR] Terjadi kegagalan pada siklus ini: {e}")

        durasi_eksekusi = time.time() - waktu_mulai
        waktu_tidur = interval_detik - durasi_eksekusi

        print(f"\n{'-'*60}")
        if waktu_tidur > 0:
            waktu_bangun = time.strftime('%H:%M:%S', time.localtime(time.time() + waktu_tidur))
            print(f"✅ [SIKLUS SELESAI] Durasi: {durasi_eksekusi:.2f} detik.")
            print(f"💤 Tidur selama {waktu_tidur/60:.2f} menit. Siklus berikutnya: {waktu_bangun}")
            await asyncio.sleep(waktu_tidur)
        else:
            print(f"⚠️ [OVERLOAD] Durasi ({durasi_eksekusi:.2f} detik) melampaui interval 45 menit!")
            print("Langsung memulai siklus baru tanpa jeda...")
            await asyncio.sleep(1)


In [14]:
if __name__ == "__main__":
    await run_ai_pipeline_periodically()


🚀 [SIKLUS AI DIMULAI: 11:06:35]

[PROSES 1/4] Menjalankan Worker 1 (Preprocessing + Smart Clustering)...

[WORKER 1] Memulai Preprocessing + Smart Clustering (Running-Centroid Mode)...
[WORKER 1] Selesai. Klaster Baru: 237, Klaster Direvisi: 30

[PROSES 2/4] Menjalankan Worker 2 (Groq AI Summarizer)...

[WORKER 2] Memulai Summarization...
[WORKER 2] Ditemukan 3 klaster untuk dirangkum.

  [V] Klaster 6787 Sukses | Token: 3130 | Waktu: 1.13s
  [V] Klaster 6834 Sukses | Token: 3089 | Waktu: 1.34s
  [V] Klaster 6887 Sukses | Token: 3881 | Waktu: 32.61s

[WORKER 2] Seluruh antrean ringkasan selesai diproses.

[PROSES 3/4] Menjalankan Worker 3 (Groq AI Sentiment Analysis)...

[WORKER 3] Memulai Sentiment Analysis (aktor diambil dari ringkasan Worker 2)...
[WORKER 3] Ditemukan 137 klaster untuk dianalisis sentimennya.

  [V] Klaster 2589 Sukses | 3 aktor | Token: 535 | Waktu: 5.88s
  [V] Klaster 5507 Sukses | 3 aktor | Token: 828 | Waktu: 5.29s
  [V] Klaster 2571 Sukses | 2 aktor | Token: 4

CancelledError: 